<a href="https://colab.research.google.com/github/balloontip/deep-learning/blob/main/chapter-08/08-05-RNN-Sequence-Classification-Project.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Goal: Build, train, and evaluate a vanilla recurrent neural network (RNN) in PyTorch that classifies numerical sequences as either increasing or decreasing. This end-to-end project demonstrates synthetic sequence generation, dataset preparation, RNN architecture, training with Cross-Entropy Loss and the Adam optimizer, gradient clipping, evaluation on unseen test data, conversion of logits to probabilities using Softmax during inference, classification accuracy measurement, and prediction confidence visualization.

In [1]:
# ==============================================================================
# 0. Imports
# ==============================================================================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Set random seeds so the example is more reproducible.
torch.manual_seed(42)
np.random.seed(42)

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

class Config:
    """Hyperparameters and settings for our RNN classification model."""
    SEQUENCE_LENGTH = 5          # Each sequence contains 5 numbers
    NUM_SAMPLES_PER_CLASS = 500  # Number of increasing/decreasing sequences
    HIDDEN_SIZE = 15             # RNN hidden state size
    BATCH_SIZE = 32              # Training batch size
    LEARNING_RATE = 0.01         # Adam optimizer learning rate
    NUM_EPOCHS = 100             # Training epochs
    TRAIN_RATIO = 0.8            # 80% training, 20% testing

config = Config()

# ==============================================================================
# 2. DATA PREPARATION
# ==============================================================================

class PatternDataset(Dataset):
    """
    Converts numerical sequences and labels into a PyTorch Dataset.

    Each sample:
        input  = sequence of numbers
        target = class label

    Class labels:
        0 = increasing pattern
        1 = decreasing pattern
    """

    def __init__(self, sequences, labels):
        self.X = torch.tensor(sequences, dtype=torch.float32).unsqueeze(-1)
        self.Y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


def generate_pattern_data(num_samples_per_class, sequence_length):
    """
    Generate simple increasing and decreasing numerical sequences.

    The goal is not to memorize exact numbers, but to learn the trend:
        increasing -> class 0
        decreasing -> class 1
    """
    X_class_A = []
    Y_class_A = []

    # Type A: Increasing sequences
    for _ in range(num_samples_per_class):
        start = np.random.rand() * 5
        step = 0.4 + np.random.rand() * 0.4
        noise = np.random.normal(0, 0.08, size=sequence_length)

        sequence = [
            start + i * step + noise[i]
            for i in range(sequence_length)
        ]

        X_class_A.append(sequence)
        Y_class_A.append(0)

    X_class_B = []
    Y_class_B = []

    # Type B: Decreasing sequences
    for _ in range(num_samples_per_class):
        start = 5 + np.random.rand() * 5
        step = 0.4 + np.random.rand() * 0.4
        noise = np.random.normal(0, 0.08, size=sequence_length)

        sequence = [
            start - i * step + noise[i]
            for i in range(sequence_length)
        ]

        X_class_B.append(sequence)
        Y_class_B.append(1)

    sequences = np.array(X_class_A + X_class_B, dtype=np.float32)
    labels = np.array(Y_class_A + Y_class_B, dtype=np.int64)

    return sequences, labels


def prepare_data():
    """
    Prepare data for RNN classification.

    Important:
    - We generate both increasing and decreasing sequences.
    - We shuffle the dataset so the model sees mixed classes during training.
    - We split the data into training and test sets.
    - We evaluate on unseen test data, not just the training data.
    """
    sequences, labels = generate_pattern_data(
        num_samples_per_class=config.NUM_SAMPLES_PER_CLASS,
        sequence_length=config.SEQUENCE_LENGTH
    )

    indices = np.arange(len(labels))
    np.random.shuffle(indices)

    sequences = sequences[indices]
    labels = labels[indices]

    train_size = int(len(labels) * config.TRAIN_RATIO)

    X_train = sequences[:train_size]
    Y_train = labels[:train_size]

    X_test = sequences[train_size:]
    Y_test = labels[train_size:]

    train_dataset = PatternDataset(X_train, Y_train)
    test_dataset = PatternDataset(X_test, Y_test)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False
    )

    print(f"Generated {len(labels)} total sequences.")
    print(f"Training samples: {len(train_dataset)}")
    print(f"Testing samples: {len(test_dataset)}")

    print(f"\nExample X batch shape: {train_dataset[0][0].shape}")
    print(f"Example Y label type: {train_dataset[0][1].dtype}")

    return train_loader, test_loader, test_dataset

# ==============================================================================
# 3. RNN MODEL ARCHITECTURE
# ==============================================================================

class RNNClassifier(nn.Module):
    """
    Vanilla RNN model for sequence classification.

    Architecture:
        Input sequence -> RNN layer -> final hidden state -> Linear output layer
    """

    def __init__(self, input_size=1, hidden_size=15, num_classes=2):
        super(RNNClassifier, self).__init__()

        self.hidden_size = hidden_size

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        # The output layer produces one raw score (logit) per class.
        # For this project:
        #   logit 0 -> increasing pattern
        #   logit 1 -> decreasing pattern
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        """
        Forward pass: sequence -> hidden states -> logits.

        Input shape:
            (batch_size, sequence_length, input_size)

        Output shape:
            (batch_size, num_classes)
        """
        batch_size = x.size(0)

        h0 = torch.zeros(
            1,
            batch_size,
            self.hidden_size,
            device=x.device
        )

        rnn_out, hn = self.rnn(x, h0)

        # Use the output from the final time step because it summarizes
        # the entire input sequence.
        final_hidden_state = rnn_out[:, -1, :]

        logits = self.fc(final_hidden_state)

        return logits

# ==============================================================================
# 4. TRAINING PIPELINE
# ==============================================================================

def calculate_accuracy(logits, labels):
    """
    Convert logits into predicted class labels and calculate accuracy.
    """
    _, predicted_labels = torch.max(logits, dim=1)
    correct = (predicted_labels == labels).sum().item()
    accuracy = correct / labels.size(0)

    return accuracy


def train_model(model, train_loader, test_loader):
    """
    Train the RNN classifier using Cross-Entropy Loss.

    CrossEntropyLoss expects:
        - raw logits from the model, not softmax probabilities
        - integer class labels, such as 0 or 1
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)

    print("\nClassifier Model Architecture:")
    print(model)

    print("\nStarting Classifier Training...")

    for epoch in range(config.NUM_EPOCHS):
        # ------------------------------
        # Training phase
        # ------------------------------
        model.train()
        train_loss = 0
        train_accuracy = 0

        for batch_data, batch_labels in train_loader:
            optimizer.zero_grad()

            logits = model(batch_data)

            loss = criterion(logits, batch_labels)

            loss.backward()

            # Gradient clipping helps prevent exploding gradients in RNNs.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            train_loss += loss.item()
            train_accuracy += calculate_accuracy(logits, batch_labels)

        train_loss = train_loss / len(train_loader)
        train_accuracy = train_accuracy / len(train_loader)

        # ------------------------------
        # Evaluation phase during training
        # ------------------------------
        model.eval()
        test_loss = 0
        test_accuracy = 0

        with torch.no_grad():
            for test_data, test_labels in test_loader:
                test_logits = model(test_data)
                loss = criterion(test_logits, test_labels)

                test_loss += loss.item()
                test_accuracy += calculate_accuracy(test_logits, test_labels)

        test_loss = test_loss / len(test_loader)
        test_accuracy = test_accuracy / len(test_loader)

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch [{epoch + 1}/{config.NUM_EPOCHS}], "
                f"Train Loss: {train_loss:.4f}, "
                f"Train Accuracy: {train_accuracy:.4f}, "
                f"Test Loss: {test_loss:.4f}, "
                f"Test Accuracy: {test_accuracy:.4f}"
            )

    print("\nClassifier Training Complete.")

# ==============================================================================
# 5. MODEL EVALUATION
# ==============================================================================

def evaluate_model(model, test_dataset, num_examples=10):
    """
    Evaluate the classifier on unseen test data.
    """
    model.eval()

    print("\n--- Classifier Evaluation on Unseen Test Data ---")

    total_correct = 0
    total_samples = len(test_dataset)

    class_names = {
        0: "Increasing",
        1: "Decreasing"
    }

    with torch.no_grad():
        for i in range(len(test_dataset)):
            sequence, label = test_dataset[i]

            sequence_input = sequence.unsqueeze(0)

            logits = model(sequence_input)

            # For display only, softmax converts logits into probabilities.
            # We do NOT use softmax before CrossEntropyLoss during training.
            probabilities = torch.softmax(logits, dim=1)

            predicted_class = torch.argmax(probabilities, dim=1).item()

            if predicted_class == label.item():
                total_correct += 1

            if i < num_examples:
                input_seq = sequence.squeeze().tolist()
                input_seq = [round(x, 2) for x in input_seq]

                actual_name = class_names[label.item()]
                predicted_name = class_names[predicted_class]
                confidence = probabilities[0, predicted_class].item()

                print(
                    f"Input: {input_seq} -> "
                    f"Actual: {actual_name}, "
                    f"Predicted: {predicted_name}, "
                    f"Confidence: {confidence:.2f}"
                )

    overall_accuracy = total_correct / total_samples

    print(f"\nOverall Accuracy on unseen test data: {overall_accuracy:.4f}")

# ==============================================================================
# 6. MAIN EXECUTION FLOW
# ==============================================================================

def main():
    """
    Complete pipeline for classifying simple numerical patterns with an RNN.

    This project demonstrates:
    - generating increasing and decreasing sequence patterns,
    - converting sequences and labels into tensors,
    - splitting data into training and test sets,
    - training an RNN classifier with Cross-Entropy Loss,
    - passing raw logits directly to the loss function,
    - using softmax only during inference for readable probabilities,
    - evaluating classification accuracy on unseen test data.
    """
    print("RNN Sequence Classification Pipeline")

    print("\n1. Preparing pattern data...")
    train_loader, test_loader, test_dataset = prepare_data()

    print("\n2. Creating RNN classifier...")
    model = RNNClassifier(
        input_size=1,
        hidden_size=config.HIDDEN_SIZE,
        num_classes=2
    )

    print("\n3. Training classifier...")
    train_model(model, train_loader, test_loader)

    print("\n4. Evaluating classifier...")
    evaluate_model(model, test_dataset)

    print("\nSequence classification complete!")

if __name__ == "__main__":
    main()


RNN Sequence Classification Pipeline

1. Preparing pattern data...
Generated 1000 total sequences.
Training samples: 800
Testing samples: 200

Example X batch shape: torch.Size([5, 1])
Example Y label type: torch.int64

2. Creating RNN classifier...

3. Training classifier...

Classifier Model Architecture:
RNNClassifier(
  (rnn): RNN(1, 15, batch_first=True)
  (fc): Linear(in_features=15, out_features=2, bias=True)
)

Starting Classifier Training...
Epoch [10/100], Train Loss: 0.0017, Train Accuracy: 1.0000, Test Loss: 0.0013, Test Accuracy: 1.0000
Epoch [20/100], Train Loss: 0.0002, Train Accuracy: 1.0000, Test Loss: 0.0002, Test Accuracy: 1.0000
Epoch [30/100], Train Loss: 0.0001, Train Accuracy: 1.0000, Test Loss: 0.0001, Test Accuracy: 1.0000
Epoch [40/100], Train Loss: 0.0001, Train Accuracy: 1.0000, Test Loss: 0.0001, Test Accuracy: 1.0000
Epoch [50/100], Train Loss: 0.0000, Train Accuracy: 1.0000, Test Loss: 0.0000, Test Accuracy: 1.0000
Epoch [60/100], Train Loss: 0.0000, Trai